# Build an eval set & score a prompt

**Session 3 · small model (`llama3.2:3b`) — the pivot**

Stop eyeballing the classifier. Build a fixed labelled set, score the prompt on it with
`repeats` so you can see the run-to-run spread, then change ONE thing and let `compare()`
say whether the move beat the noise.

Two iterations, two different outcomes:
1. Fix the **output format** — a large, unambiguous win (the first score is low because you
   can't parse the answer, not because the model is wrong).
2. Add label **definitions** — a change that lands *inside* the noise. That is also a result.

We use the **support-ticket router** from Session 2 (the 3B model is ~99% on the sentiment
set, so there is nothing to iterate on there).

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
from utils import ask, SMALL_MODEL
from eval import load_cases, run_eval, print_report, compare, exact


### Score the first prompt — strictly

Score the **raw** model output against the gold label with `exact` (no cleanup). This is the
honest number: if your pipeline can't parse it, it's wrong.

In [ ]:
cases = load_cases("../eval/datasets/support_tickets.jsonl")
LABELS = ["billing", "bug", "other"]

NAIVE = 'Classify this support ticket into billing, bug, or other.\n\n"{t}"'

def raw_classifier(prompt_tmpl):
    def classify(text):
        return ask(prompt_tmpl.format(t=text), model=SMALL_MODEL).strip().lower().strip('."\' ')
    return classify

report = run_eval(cases, raw_classifier(NAIVE), scorer=exact, repeats=3)
print_report(report)

### Iteration 1 — fix the output format

The model knows the answer; it just won't say it in one word. Add a format rule and three
examples. Score the same way and `compare()`.

### Worked example

`NAIVE` vs `DISCIPLINED` (format rule + few-shot), strict scoring, then a per-case report of
what still fails.

In [ ]:
DISCIPLINED = (
    'Classify the support ticket as billing, bug, or other.\n'
    'Reply with ONE lowercase word and nothing else.\n\n'
    'Ticket: "I cannot log in since the update" -> bug\n'
    'Ticket: "Refund me for the double charge" -> billing\n'
    'Ticket: "What are your office hours?" -> other\n\n'
    'Ticket: "{t}" ->'
)

compare(
    cases,
    raw_classifier(NAIVE),
    raw_classifier(DISCIPLINED),
    labels=("naive", "format rule + few-shot"),
    scorer=exact,
    repeats=3,
)
print()
print_report(run_eval(cases, raw_classifier(DISCIPLINED), scorer=exact, repeats=1))


### Iteration 2 — add label definitions

Switch to a tolerant scorer (`label_in`: the label appears anywhere in the reply) so we're
measuring *accuracy* now, not format. Does spelling out what each label means help?

In [ ]:
def label_in(output, expected):
    return expected in output.lower()

WITH_DEFS = (
    'Classify the support ticket as billing, bug, or other.\n'
    '- billing: charges, invoices, receipts, refunds, payment methods, cancellations\n'
    '- bug: a feature is broken, wrong, or behaves unexpectedly\n'
    '- other: how-to questions, account settings, general product questions\n'
    'Reply with ONE lowercase word and nothing else.\n\n'
    'Ticket: "I cannot log in since the update" -> bug\n'
    'Ticket: "Refund me for the double charge" -> billing\n'
    'Ticket: "What are your office hours?" -> other\n\n'
    'Ticket: "{t}" ->'
)

compare(
    cases,
    raw_classifier(DISCIPLINED),
    raw_classifier(WITH_DEFS),
    labels=("no definitions", "with definitions"),
    scorer=label_in,
    repeats=5,
)


## Your turn - vary the example

1. Look at the FAIL lines. Add ONE few-shot example targeting the most common failure, re-run
   `compare()`. Did overall accuracy rise past the noise, or did you trade one failure for another?
2. If `compare()` says INCONCLUSIVE, bump `repeats` to 8. Does the verdict firm up, or is the
   change genuinely too small to matter?
3. Record the before/after mean, the spread, and the verdict in your commit message and PR.
